<a href="https://colab.research.google.com/github/Ankoor11/Fine-Tuning-BERT-for-Drug-Related-Text-Classification/blob/main/BERT_Drug_Classification_done_on_Colab__.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 📋 STEP 1: SETUP AND MOUNT GOOGLE DRIVE
# Mount Google Drive to access your files
from google.colab import drive
drive.mount('/content/drive')

# Create working directories
import os
os.makedirs('/content/dataset', exist_ok=True)
os.makedirs('/content/models', exist_ok=True)
os.makedirs('/content/results', exist_ok=True)

print("✓ Google Drive mounted")
print("✓ Working directories created")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Google Drive mounted
✓ Working directories created


In [ ]:
# 📦 STEP 2: INSTALL REQUIRED LIBRARIES
!pip install torch transformers datasets scikit-learn pandas numpy -q

print("\n✓ PyTorch installed")
print("✓ Transformers installed")
print("✓ Datasets library installed")
print("✓ All dependencies ready!")


✓ PyTorch installed
✓ Transformers installed
✓ Datasets library installed
✓ All dependencies ready!


In [ ]:
# ⚙️ STEP 3: CHECK SYSTEM CONFIGURATION
import torch

print("=" * 60)
print("SYSTEM CONFIGURATION")
print("=" * 60)

# Check GPU
if torch.cuda.is_available():
    print(f"✓ GPU Available: {torch.cuda.get_device_name(0)}")
    print(f"✓ GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    device = "cuda"
else:
    print("⚠ GPU NOT available - will use CPU (slower)")
    device = "cpu"

# Check PyTorch version
print(f"✓ PyTorch Version: {torch.__version__}")

# Set device
device = torch.device(device)
print(f"✓ Device set to: {device}")

SYSTEM CONFIGURATION
✓ GPU Available: Tesla T4
✓ GPU Memory: 15.64 GB
✓ PyTorch Version: 2.10.0+cu128
✓ Device set to: cuda


In [ ]:
# 📥 STEP 4: UPLOAD DATASET FILES
# OPTION A: Upload from your computer
from google.colab import files

print("Uploading dataset files...")
print("Select these files:")
print("  - bert_train.jsonl")
print("  - bert_val.jsonl")
print("  - bert_test.jsonl")
print()

uploaded = files.upload()

# Move files to dataset directory
import shutil
for filename in uploaded.keys():
    shutil.move(filename, f'/content/dataset/{filename}')
    print(f"✓ Moved {filename} to /content/dataset/")

Uploading dataset files...
Select these files:
  - bert_train.jsonl
  - bert_val.jsonl
  - bert_test.jsonl



Saving bert_test.jsonl to bert_test.jsonl
Saving bert_val.jsonl to bert_val.jsonl
Saving bert_train.jsonl to bert_train.jsonl
✓ Moved bert_test.jsonl to /content/dataset/
✓ Moved bert_val.jsonl to /content/dataset/
✓ Moved bert_train.jsonl to /content/dataset/


In [ ]:
import json
import os

dataset_path = '/content/dataset'

print("=" * 60)
print("DATASET VERIFICATION")
print("=" * 60)

for filename in ['bert_train.jsonl', 'bert_val.jsonl', 'bert_test.jsonl']:
    filepath = os.path.join(dataset_path, filename)

    if os.path.exists(filepath):
        with open(filepath, 'r') as f:
            lines = f.readlines()

        sample = json.loads(lines[0])

        print(f"\n✓ {filename}")
        print(f"  Samples: {len(lines)}")
        print(f"  Keys: {list(sample.keys())}")
        print(f"  Text: {sample['text'][:80]}...")
        print(f"  Label: {sample['label']}")  # ← fixed: was 'keywords'
    else:
        print(f"\n✗ {filename} NOT FOUND")

DATASET VERIFICATION

✓ bert_train.jsonl
  Samples: 4000
  Keys: ['text', 'label']
  Text: Cooking yogurt is my weekend ritual...
  Label: 0

✓ bert_val.jsonl
  Samples: 500
  Keys: ['text', 'label']
  Text: crack addiction is a serious problem...
  Label: 1

✓ bert_test.jsonl
  Samples: 500
  Keys: ['text', 'label']
  Text: The doctor prescribed spice for my muscle ache...
  Label: 1


In [ ]:
# 🔄 STEP 6: LOAD DATASET
from datasets import load_dataset
import os

dataset_path = '/content/dataset'

print("Loading dataset from JSONL files...")

# Load dataset
dataset = load_dataset('json', data_files={
    'train': os.path.join(dataset_path, 'bert_train.jsonl'),
    'validation': os.path.join(dataset_path, 'bert_val.jsonl'),
    'test': os.path.join(dataset_path, 'bert_test.jsonl')
})

print("✓ Dataset loaded successfully!")
print(f"\nDataset structure:")
print(f"  Train: {len(dataset['train'])} samples")
print(f"  Val: {len(dataset['validation'])} samples")
print(f"  Test: {len(dataset['test'])} samples")

# Show sample
print(f"\nSample record:")
sample = dataset['train'][0]
for key, value in sample.items():
    if key == 'text':
        print(f"  {key}: {value[:100]}...")
    else:
        print(f"  {key}: {value}")

Loading dataset from JSONL files...


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

✓ Dataset loaded successfully!

Dataset structure:
  Train: 4000 samples
  Val: 500 samples
  Test: 500 samples

Sample record:
  text: Cooking yogurt is my weekend ritual...
  label: 0


In [ ]:
# 🔤 STEP 7: TOKENIZE DATASET
from transformers import AutoTokenizer
import numpy as np

print("Loading tokenizer...")

# Load tokenizer
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

print(f"✓ Tokenizer loaded: {model_name}")

# Create tokenization function
def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        padding="max_length",
        truncation=True,
        max_length=512
    )

print("\nTokenizing dataset (this may take 1-2 minutes)...")

# Tokenize all datasets
tokenized_datasets = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=['text'],  # ← FIXED: only 'text' exists to remove
    batch_size=32
)

print("✓ Tokenization complete!")

# Rename label column
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")

print(f"\nTokenized dataset structure:")
print(f"  Train: {len(tokenized_datasets['train'])} samples")
print(f"  Val: {len(tokenized_datasets['validation'])} samples")
print(f"  Test: {len(tokenized_datasets['test'])} samples")

# Show token example
sample = tokenized_datasets['train'][0]
print(f"\nTokenized sample:")
print(f"  Input IDs shape: {np.array(sample['input_ids']).shape}")
print(f"  Attention mask shape: {np.array(sample['attention_mask']).shape}")
print(f"  Label: {sample['labels']}")

Loading tokenizer...
✓ Tokenizer loaded: bert-base-uncased

Tokenizing dataset (this may take 1-2 minutes)...


Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

✓ Tokenization complete!

Tokenized dataset structure:
  Train: 4000 samples
  Val: 500 samples
  Test: 500 samples

Tokenized sample:
  Input IDs shape: (512,)
  Attention mask shape: (512,)
  Label: 0


In [ ]:
# 🧠 STEP 8: LOAD BERT MODEL
from transformers import AutoModelForSequenceClassification
import torch

print("Loading BERT model...")

model_name = "bert-base-uncased"

# Load model
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    id2label={0: "not_drug_related", 1: "drug_related"},
    label2id={"not_drug_related": 0, "drug_related": 1}
)

# Move to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print(f"✓ Model loaded and moved to {device}")

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\nModel Parameters:")
print(f"  Total: {total_params:,}")
print(f"  Trainable: {trainable_params:,}")

print(f"\nLabel mappings:")
print(f"  0 → {model.config.id2label[0]}")
print(f"  1 → {model.config.id2label[1]}")

Loading BERT model...


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✓ Model loaded and moved to cuda

Model Parameters:
  Total: 109,483,778
  Trainable: 109,483,778

Label mappings:
  0 → not_drug_related
  1 → drug_related


In [ ]:
# ⚙️ STEP 9: SETUP TRAINING CONFIGURATION (FIXED)
from transformers import TrainingArguments, DataCollatorWithPadding

print("Setting up training configuration...")

# Training arguments
training_args = TrainingArguments(
    output_dir="/content/models/bert_drug_classifier",

    # Learning parameters
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,

    # Optimization
    weight_decay=0.01,
    warmup_steps=500,

    # Evaluation and saving
    eval_strategy="epoch",              # ← CHANGED from evaluation_strategy
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,

    # Logging
    logging_steps=50,
    logging_dir="/content/results/logs",

    # Other
    seed=42,
    push_to_hub=False,
    report_to="none",
)

print("✓ Training arguments configured")

# Data collator for padding
data_collator = DataCollatorWithPadding(tokenizer)
print("✓ Data collator ready")

print("\nTraining Configuration:")
print(f"  Learning Rate: {training_args.learning_rate}")
print(f"  Batch Size: {training_args.per_device_train_batch_size}")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Weight Decay: {training_args.weight_decay}")
print(f"  Warmup Steps: {training_args.warmup_steps}")

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Setting up training configuration...
✓ Training arguments configured
✓ Data collator ready

Training Configuration:
  Learning Rate: 2e-05
  Batch Size: 16
  Epochs: 3
  Weight Decay: 0.01
  Warmup Steps: 500


In [ ]:
# 📊 STEP 10: SETUP EVALUATION METRICS
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import numpy as np

print("Setting up evaluation metrics...")

def compute_metrics(eval_pred):
    """Compute metrics for evaluation"""
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)

    # Calculate metrics
    accuracy = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average='weighted'
    )

    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
    }

print("✓ Metrics function defined")
print("\nMetrics to track:")
print("  - Accuracy")
print("  - Precision")
print("  - Recall")
print("  - F1-Score")

Setting up evaluation metrics...
✓ Metrics function defined

Metrics to track:
  - Accuracy
  - Precision
  - Recall
  - F1-Score


In [ ]:
# 🚂 STEP 11: INITIALIZE TRAINER (FIXED)
from transformers import Trainer

print("Initializing Trainer...")

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,  # ← tokenizer line removed
)

print("✓ Trainer initialized")
print(f"\nTraining will run:")
print(f"  - {len(tokenized_datasets['train'])} training samples")
print(f"  - {len(tokenized_datasets['validation'])} validation samples")
print(f"  - For {training_args.num_train_epochs} epochs")
print(f"  - On device: {trainer.args.device}")

Initializing Trainer...
✓ Trainer initialized

Training will run:
  - 4000 training samples
  - 500 validation samples
  - For 3 epochs
  - On device: cuda:0


In [ ]:
# ⭐ STEP 12: START TRAINING (30-60 MINUTES)
import time

print("=" * 70)
print("STARTING TRAINING")
print("=" * 70)
print(f"Time: {time.strftime('%Y-%m-%d %H:%M:%S')}")
print()

# Train
train_result = trainer.train()

print("\n" + "=" * 70)
print("TRAINING COMPLETE")
print("=" * 70)
print(f"Time: {time.strftime('%Y-%m-%d %H:%M:%S')}")

# Print training results
print(f"\nTraining Results:")
print(f"  Training Loss: {train_result.training_loss:.4f}")
print(f"  Training Time: {train_result.training_steps} steps")

STARTING TRAINING
Time: 2026-02-27 20:21:18



Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.000316,0.000196,1.000000,1.000000,1.000000,1.000000
2,0.000157,0.000093,1.000000,1.000000,1.000000,1.000000
3,0.000105,0.000067,1.000000,1.000000,1.000000,1.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La


TRAINING COMPLETE
Time: 2026-02-27 20:42:28

Training Results:
  Training Loss: 0.0002


AttributeError: 'TrainOutput' object has no attribute 'training_steps'

In [ ]:
# 📈 STEP 13: EVALUATE ON TEST SET
print("=" * 70)
print("EVALUATING ON TEST SET")
print("=" * 70)
print()

# Evaluate on test set
test_results = trainer.evaluate(tokenized_datasets["test"])

print("Test Set Results:")
print("=" * 70)
for metric, value in test_results.items():
    if 'loss' in metric:
        print(f"  {metric}: {value:.4f}")
    else:
        print(f"  {metric}: {value:.4f}")

# Save results to file
import json
results_path = '/content/results/test_results.json'
with open(results_path, 'w') as f:
    json.dump(test_results, f, indent=2)

print(f"\n✓ Results saved to {results_path}")

EVALUATING ON TEST SET



Test Set Results:
  eval_loss: 0.0002
  eval_accuracy: 1.0000
  eval_precision: 1.0000
  eval_recall: 1.0000
  eval_f1: 1.0000
  eval_runtime: 16.5359
  eval_samples_per_second: 30.2370
  eval_steps_per_second: 1.9350
  epoch: 3.0000

✓ Results saved to /content/results/test_results.json


In [ ]:
# 💾 STEP 14: SAVE MODEL
import os

print("=" * 70)
print("SAVING MODEL")
print("=" * 70)

output_dir = "/content/models/bert_drug_classifier_final"
os.makedirs(output_dir, exist_ok=True)

# Save model
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

print(f"✓ Model saved to {output_dir}")

print(f"\nFiles saved:")
for filename in os.listdir(output_dir):
    filepath = os.path.join(output_dir, filename)
    size = os.path.getsize(filepath) / (1024 * 1024)
    print(f"  - {filename} ({size:.2f} MB)")

SAVING MODEL


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✓ Model saved to /content/models/bert_drug_classifier_final

Files saved:
  - config.json (0.00 MB)
  - tokenizer.json (0.68 MB)
  - tokenizer_config.json (0.00 MB)
  - model.safetensors (417.67 MB)


In [ ]:
# 🔄 STEP 15: BACKUP TO GOOGLE DRIVE
import shutil
import os

print("Copying model to Google Drive...")

# Define paths
drive_output = '/content/drive/MyDrive/bert_drug_classifier_final'
local_output = '/content/models/bert_drug_classifier_final'

# Create directory in Drive
os.makedirs(drive_output, exist_ok=True)

# Copy all files
for filename in os.listdir(local_output):
    src = os.path.join(local_output, filename)
    dst = os.path.join(drive_output, filename)
    if os.path.isfile(src):
        shutil.copy2(src, dst)
        print(f"  ✓ Copied {filename}")

print(f"\n✓ Model backed up to Google Drive: {drive_output}")

Copying model to Google Drive...
  ✓ Copied config.json
  ✓ Copied tokenizer.json
  ✓ Copied tokenizer_config.json
  ✓ Copied model.safetensors

✓ Model backed up to Google Drive: /content/drive/MyDrive/bert_drug_classifier_final


In [ ]:
# 🔮 STEP 16: CREATE INFERENCE FUNCTION
import torch
import torch.nn.functional as F
from transformers import AutoModelForSequenceClassification, AutoTokenizer

class DrugClassifier:
    """Drug classification using fine-tuned BERT"""

    def __init__(self, model_path):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model = AutoModelForSequenceClassification.from_pretrained(model_path)
        self.tokenizer = AutoTokenizer.from_pretrained(model_path)
        self.model.to(self.device)
        self.model.eval()

    def classify(self, text):
        """Classify a single text"""
        # Tokenize
        inputs = self.tokenizer(
            text,
            return_tensors="pt",
            max_length=512,
            truncation=True,
            padding=True
        ).to(self.device)

        # Predict
        with torch.no_grad():
            outputs = self.model(**inputs)
            logits = outputs.logits

        # Get probabilities
        probabilities = F.softmax(logits, dim=1)[0].cpu().numpy()
        predicted_class = int(torch.argmax(logits, dim=1)[0].cpu().numpy())

        # Labels
        labels = {0: "not_drug_related", 1: "drug_related"}

        return {
            "text": text[:100] + "..." if len(text) > 100 else text,
            "label": labels[predicted_class],
            "confidence": float(probabilities[predicted_class]),
            "probabilities": {
                "not_drug_related": float(probabilities[0]),
                "drug_related": float(probabilities[1])
            }
        }

print("✓ Classifier class defined")

✓ Classifier class defined


In [ ]:
# 🧪 STEP 17: TEST INFERENCE
print("Loading fine-tuned model for inference...")

model_path = "/content/models/bert_drug_classifier_final"
classifier = DrugClassifier(model_path)

print("✓ Model loaded successfully!")

# Test cases
test_cases = [
    "I took xanax for my anxiety disorder",
    "I love watching movies with my family",
    "Opioid addiction is difficult to overcome",
    "The weather was beautiful today",
    "Heroin withdrawal symptoms are severe",
    "I had chocolate cake for dessert",
]

print("\n" + "=" * 70)
print("INFERENCE EXAMPLES")
print("=" * 70)

for i, text in enumerate(test_cases, 1):
    result = classifier.classify(text)

    print(f"\n{i}. Text: {text}")
    print(f"   Label: {result['label']}")
    print(f"   Confidence: {result['confidence']:.4f}")
    print(f"   Not Drug: {result['probabilities']['not_drug_related']:.4f}")
    print(f"   Drug: {result['probabilities']['drug_related']:.4f}")

Loading fine-tuned model for inference...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✓ Model loaded successfully!

INFERENCE EXAMPLES

1. Text: I took xanax for my anxiety disorder
   Label: drug_related
   Confidence: 0.9998
   Not Drug: 0.0002
   Drug: 0.9998

2. Text: I love watching movies with my family
   Label: not_drug_related
   Confidence: 0.9998
   Not Drug: 0.9998
   Drug: 0.0002

3. Text: Opioid addiction is difficult to overcome
   Label: drug_related
   Confidence: 0.9998
   Not Drug: 0.0002
   Drug: 0.9998

4. Text: The weather was beautiful today
   Label: not_drug_related
   Confidence: 0.9998
   Not Drug: 0.9998
   Drug: 0.0002

5. Text: Heroin withdrawal symptoms are severe
   Label: drug_related
   Confidence: 0.9998
   Not Drug: 0.0002
   Drug: 0.9998

6. Text: I had chocolate cake for dessert
   Label: not_drug_related
   Confidence: 0.9998
   Not Drug: 0.9998
   Drug: 0.0002
